# 🧠 Train & Register Stock Forecast Ensemble Model
Trains LightGBM + RandomForest + GradientBoosting ensemble with walk-forward cross-validation.
Registers to Unity Catalog as `riskbricks.models.stock_forecast_ensemble` with production alias.

**Architecture:**
| Component | Config |
|-----------|--------|
| LightGBM | num_leaves=8, lr=0.1, n_estimators=50 |
| RandomForest | n_estimators=100, max_depth=5 |
| GradientBoosting | n_estimators=50, max_depth=3, lr=0.1 |
| Ensemble | Soft vote (avg probabilities) |
| Confidence Filter | >40% = high-confidence (>80% or <20% prob) |

**Training Data:** `riskbricks.silver.ml_training_features` (408 samples × 17 features)
**Walk-Forward:** Train on days 1..N-1, predict day N, rotate.

In [0]:
%pip install lightgbm scikit-learn mlflow -q
dbutils.library.restartPython()

In [0]:
# ── Import centralized config ────────────────────────────────────────
import sys, os
_nb  = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_root = "/Workspace" + _nb[:_nb.find("/notebooks/")]
sys.path.insert(0, _root)
from config import CATALOG as _CFG_CATALOG, MODEL_NAME as _CFG_MODEL, CURATED_FEATURES, LGB_PARAMS, RF_PARAMS, GB_PARAMS

import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

dbutils.widgets.text("catalog", "riskbricks")
CATALOG = dbutils.widgets.get("catalog").strip()
MODEL_NAME = f"{CATALOG}.models.stock_forecast_ensemble"
# Dynamic experiment path based on current user
current_user = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_PATH = f"/Users/{current_user}/riskbricks_stock_forecast"

# ML Hyper-parameters from config
# (LGB_PARAMS, RF_PARAMS, GB_PARAMS already imported)

mlflow.set_experiment(EXPERIMENT_PATH)
print(f"\u2705 Config loaded | Model: {MODEL_NAME} | Features: {len(CURATED_FEATURES)}")
print(f"   LGB: {LGB_PARAMS}")
print(f"   RF:  {RF_PARAMS}")
print(f"   GB:  {GB_PARAMS}")

# --- Production Logging ---
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("riskbricks.ml_training")
logger.info(f"Starting ML Ensemble Training — catalog={CATALOG}")

In [0]:
# ── Load Training Data ─────────────────────────────────────────
df = spark.table(f"{CATALOG}.silver.ml_training_features").toPandas()
print(f"Loaded {len(df)} samples from {CATALOG}.silver.ml_training_features")
print(f"Columns: {list(df.columns)}")
if 'date' in df.columns:    print(f"Date range: {df['date'].min()} → {df['date'].max()}")

print(f"Symbols: {df['symbol'].nunique()} unique")

# Create binary target: 1 = UP, 0 = DOWN
if 'actual_direction' in df.columns:
    df['target'] = (df['actual_direction'] == 'UP').astype(int)
elif 'next_day_return' in df.columns:
    df['target'] = (df['next_day_return'] > 0).astype(int)
else:
    # Compute from next-day close
    df = df.sort_values(['symbol', 'date'])
    df['next_close'] = df.groupby('symbol')['last_close'].shift(-1)
    df['target'] = (df['next_close'] > df['last_close']).astype(int)
    df = df.dropna(subset=['target'])
    df['target'] = df['target'].astype(int)

print(f"\nTarget distribution: UP={df['target'].sum()} ({df['target'].mean():.1%}), DOWN={(1-df['target']).sum()} ({1-df['target'].mean():.1%})")
print(f"Training samples after filtering: {len(df)}")

# Ensure all features exist, fill missing with 0
for feat in CURATED_FEATURES:
    if feat not in df.columns:
        df[feat] = 0
        print(f"  ⚠️ Missing feature '{feat}' — filled with 0")

X = df[CURATED_FEATURES].fillna(0).values
y = df['target'].values
dates = df['date'].values if 'date' in df.columns else df.index.values
print(f"\n✅ Feature matrix: {X.shape[0]} samples × {X.shape[1]} features")

In [0]:
# ── Walk-Forward Cross-Validation ────────────────────────────────
# Train on days 1..N-1, predict day N for each unique date
unique_dates = sorted(df['date'].unique())
print(f"Walk-forward validation across {len(unique_dates)} trading days:")
print(f"Dates: {[str(d)[:10] for d in unique_dates]}\n")

all_preds = []
all_probs = []
all_actuals = []
all_confs = []
day_results = []

for i in range(1, len(unique_dates)):  # Skip first day (need history)
    train_dates = unique_dates[:i]
    test_date = unique_dates[i]
    
    train_mask = df['date'].isin(train_dates)
    test_mask = df['date'] == test_date
    
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    if len(X_train) < 10 or len(X_test) == 0:
        continue
    
    # Train 3 models
    lgb_model = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_model.fit(X_train, y_train)
    lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
    
    rf_model = RandomForestClassifier(**RF_PARAMS)
    rf_model.fit(X_train, y_train)
    rf_prob = rf_model.predict_proba(X_test)[:, 1]
    
    gb_model = GradientBoostingClassifier(**GB_PARAMS)
    gb_model.fit(X_train, y_train)
    gb_prob = gb_model.predict_proba(X_test)[:, 1]
    
    # Ensemble: average probabilities
    ensemble_prob = (lgb_prob + rf_prob + gb_prob) / 3
    ensemble_pred = (ensemble_prob > 0.5).astype(int)
    confidence = np.abs(ensemble_prob - 0.5) * 2
    
    acc = accuracy_score(y_test, ensemble_pred)
    hi_conf_mask = confidence > 0.4
    hi_conf_acc = accuracy_score(y_test[hi_conf_mask], ensemble_pred[hi_conf_mask]) if hi_conf_mask.sum() > 0 else 0
    
    all_preds.extend(ensemble_pred)
    all_probs.extend(ensemble_prob)
    all_actuals.extend(y_test)
    all_confs.extend(confidence)
    
    day_results.append({
        'date': str(test_date)[:10], 'n_stocks': len(y_test),
        'accuracy': acc, 'hi_conf_acc': hi_conf_acc,
        'hi_conf_n': int(hi_conf_mask.sum()), 'up_pct': y_test.mean()
    })
    direction = '🟢' if y_test.mean() > 0.5 else '🔴'
    print(f"  {direction} {str(test_date)[:10]}: {acc:.1%} ({len(y_test)} stocks) | Hi-conf: {hi_conf_acc:.1%} ({int(hi_conf_mask.sum())} trades)")

all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)
all_confs = np.array(all_confs)

overall_acc = accuracy_score(all_actuals, all_preds)
hi_mask = all_confs > 0.4
hi_conf_overall = accuracy_score(all_actuals[hi_mask], all_preds[hi_mask]) if hi_mask.sum() > 0 else 0

print(f"\n{'='*60}")
print(f"✅ Overall Walk-Forward Accuracy: {overall_acc:.1%}")
print(f"✅ High-Confidence (>40%): {hi_conf_overall:.1%} ({hi_mask.sum()}/{len(all_preds)} trades)")
print(f"{'='*60}")

In [0]:
# ── Ensemble PyFunc Wrapper ──────────────────────────────────────
# Custom MLflow PyFunc that wraps all 3 models
class StockForecastEnsemble(mlflow.pyfunc.PythonModel):
    """Ensemble of LightGBM + RandomForest + GradientBoosting for stock direction prediction."""
    
    def __init__(self, lgb_model, rf_model, gb_model, feature_names):
        self.lgb_model = lgb_model
        self.rf_model = rf_model
        self.gb_model = gb_model
        self.feature_names = feature_names
    
    def predict(self, context, model_input, params=None):
        """Predict stock direction with ensemble voting.
        
        Returns DataFrame with: direction, probability_up, confidence, lgb_prob, rf_prob, gb_prob
        """
        if isinstance(model_input, pd.DataFrame):
            # Extract only the features we need
            available = [f for f in self.feature_names if f in model_input.columns]
            X = model_input[available].fillna(0).values
            # Pad missing features with 0
            if len(available) < len(self.feature_names):
                full_X = np.zeros((len(model_input), len(self.feature_names)))
                for i, f in enumerate(self.feature_names):
                    if f in model_input.columns:
                        full_X[:, i] = model_input[f].fillna(0).values
                X = full_X
        else:
            X = np.array(model_input)
        
        lgb_prob = self.lgb_model.predict_proba(X)[:, 1]
        rf_prob = self.rf_model.predict_proba(X)[:, 1]
        gb_prob = self.gb_model.predict_proba(X)[:, 1]
        
        ensemble_prob = (lgb_prob + rf_prob + gb_prob) / 3
        confidence = np.abs(ensemble_prob - 0.5) * 2
        direction = np.where(ensemble_prob > 0.5, "UP", "DOWN")
        
        return pd.DataFrame({
            "direction": direction,
            "probability_up": np.round(ensemble_prob, 4),
            "confidence": np.round(confidence, 4),
            "lgb_prob": np.round(lgb_prob, 4),
            "rf_prob": np.round(rf_prob, 4),
            "gb_prob": np.round(gb_prob, 4),
        })

print("✅ StockForecastEnsemble PyFunc class defined")

In [0]:
# ── Train Final Models on ALL Data + Log to MLflow ──────────────
with mlflow.start_run(run_name="ensemble_retrain") as run:
    # Train on full dataset
    lgb_final = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_final.fit(X, y)
    
    rf_final = RandomForestClassifier(**RF_PARAMS)
    rf_final.fit(X, y)
    
    gb_final = GradientBoostingClassifier(**GB_PARAMS)
    gb_final.fit(X, y)
    
    # Log hyperparameters
    mlflow.log_params({
        "lgb_num_leaves": LGB_PARAMS["num_leaves"],
        "lgb_lr": LGB_PARAMS["learning_rate"],
        "lgb_n_estimators": LGB_PARAMS["n_estimators"],
        "lgb_min_child_samples": LGB_PARAMS["min_child_samples"],
        "rf_n_estimators": RF_PARAMS["n_estimators"],
        "rf_max_depth": RF_PARAMS["max_depth"],
        "rf_min_samples_leaf": RF_PARAMS["min_samples_leaf"],
        "gb_n_estimators": GB_PARAMS["n_estimators"],
        "gb_max_depth": GB_PARAMS["max_depth"],
        "gb_lr": GB_PARAMS["learning_rate"],
        "n_features": len(CURATED_FEATURES),
        "n_training_samples": len(X),
        "n_symbols": df['symbol'].nunique() if 'symbol' in df.columns else 0,
        "n_trading_days": len(unique_dates),
    })
    
    # Log walk-forward metrics
    mlflow.log_metrics({
        "walkforward_accuracy": round(overall_acc, 4),
        "walkforward_hi_conf_accuracy": round(hi_conf_overall, 4),
        "walkforward_hi_conf_trades": int(hi_mask.sum()),
        "walkforward_total_trades": len(all_preds),
    })
    
    # Log feature importance
    importance = pd.DataFrame({
        'feature': CURATED_FEATURES,
        'lgb_importance': lgb_final.feature_importances_,
        'rf_importance': rf_final.feature_importances_,
        'gb_importance': gb_final.feature_importances_,
    })
    importance['avg_importance'] = importance[['lgb_importance','rf_importance','gb_importance']].mean(axis=1)
    importance = importance.sort_values('avg_importance', ascending=False)
    mlflow.log_text(importance.to_string(index=False), "feature_importance.txt")
    print("\n📊 Feature Importance (top 10):")
    print(importance.head(10).to_string(index=False))
    
    # Log day-by-day results
    day_df = pd.DataFrame(day_results)
    mlflow.log_text(day_df.to_string(index=False), "daily_walkforward_results.txt")
    
    # Create ensemble wrapper
    ensemble = StockForecastEnsemble(lgb_final, rf_final, gb_final, CURATED_FEATURES)
    
    # Create signature from sample
    sample_input = pd.DataFrame([dict(zip(CURATED_FEATURES, X[0]))], columns=CURATED_FEATURES)
    sample_output = ensemble.predict(None, sample_input)
    signature = infer_signature(sample_input, sample_output)
    
    # Log the PyFunc model
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ensemble,
        signature=signature,
        input_example=sample_input,
        pip_requirements=["lightgbm", "scikit-learn", "pandas", "numpy"],
        registered_model_name=MODEL_NAME,
    )
    
    run_id = run.info.run_id
    print(f"\n✅ Model logged to MLflow")
    print(f"   Run ID: {run_id}")
    print(f"   Registered: {MODEL_NAME}")
    print(f"   Walk-Forward Accuracy: {overall_acc:.1%}")
    print(f"   Hi-Conf Accuracy: {hi_conf_overall:.1%}")

In [0]:
# ── Set Production Alias ─────────────────────────────────────────
from mlflow import MlflowClient

client = MlflowClient()

# Get latest version
latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if latest_versions:
    latest_version = max(latest_versions, key=lambda v: int(v.version))
    version_num = latest_version.version
    
    # Set production alias
    client.set_registered_model_alias(MODEL_NAME, "production", version_num)
    print(f"✅ Set 'production' alias → {MODEL_NAME} v{version_num}")
    print(f"   Model URI: models:/{MODEL_NAME}@production")
    print(f"   Run ID: {latest_version.run_id}")
else:
    print("❌ No model versions found")

In [0]:
# ── Validate: Load Registered Model & Test ────────────────────
import mlflow

loaded = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@production")

# Test with sample from training data
sample = pd.DataFrame([dict(zip(CURATED_FEATURES, X[0]))], columns=CURATED_FEATURES)
result = loaded.predict(sample)
print("🧪 Validation — loaded model from UC and predicted on 1 sample:")
print(result.to_string(index=False))

# Test on full training set for sanity
full_test = pd.DataFrame(X, columns=CURATED_FEATURES)
full_results = loaded.predict(full_test)
full_acc = accuracy_score(y, (full_results['probability_up'] > 0.5).astype(int))
print(f"\n✅ Full training set accuracy (in-sample): {full_acc:.1%}")
print(f"✅ Production model validated successfully!")
print(f"   URI: models:/{MODEL_NAME}@production")

### Active Notebooks (21 files)

**jobs/** — `daily_data_refresh`, `daily_gdelt_refresh`

**ingestion/** — `ml_data_ingestion`, `stocks/ingest_stocks_and_macros_data`, `stocks/bronze_to_gold_daily_stocks_macros`, `forecast/build_forecast_features_daily`, `gdelt/bronze_ingest_gdelt`, `rss/bronze_ingest_rss_news`, `portfolio/ingest_setup_multi_manager_portfolios`

**gold/** — `analytics/create_risk_analytics`, `analytics/build_portfolio_manager_outputs`, `forecast/generate_stock_forecasts`, `forecast/evaluate_stock_forecasts`, `forecast/train_forecast_model`

**training/** — `train_register_ensemble_model`

**pipelines/** — `news_to_forecasts_pipeline`, `ml_feature_pipeline`

**agents/** — `riskbricks_agent`, `01_register_uc_tools`, `02_create_agent`, `03_deploy_agent`


In [0]:
# Archive/cleanup already completed during architecture review.
# Old notebooks moved from 00_bronze/, 02_silver/, 03_gold/, 09_adhoc/
# to notebooks/_archive/stale_notebooks/
# See DATA_ARCHITECTURE.md for current structure.
print('✅ Archive cleanup already completed')
